In [50]:
from nnAudio.features.cqt import CQT
import torch
import math
from matplotlib import pyplot as plt
import numpy as np
import lightning as L

from murenn import MuReNNDirect


In [51]:
HYPERPARAMS = {
    "sr": 44100,
    "seg_length": 2**14,
    "bins_per_octave": 8,
    "n_octave": 8,
    "n_bins": 64,
    "fmin": 22050 // 2**8,
    "fmax": 22050,
    "hop_length": 2**8,
    "n_samples": 1000,
}

In [52]:
class SineData(torch.utils.data.Dataset):
    def __init__(self, spec):
        super().__init__()
        self.spec = spec
        self.cqt_params = {
            "sr": spec["sr"],
            "bins_per_octave": spec["bins_per_octave"],
            "hop_length": spec["hop_length"],
            "n_bins": spec["n_bins"]
        }
        self.freqs = np.logspace(
            np.log10(spec["fmin"]/2),
            np.log10(spec["fmax"]/2),
            spec["n_samples"],
            endpoint=False,
        )
        self.closure = CQT(**self.cqt_params)
        self.seg_length = spec["seg_length"]
        self.sr = spec["sr"]
    
    def __getitem__(self, idx):
        freq = self.freqs[idx]
        t = torch.arange(0, self.seg_length/self.sr, 1/self.sr)
        x = torch.sin(2*math.pi*freq*t)
        y = self.closure(x).squeeze()
        return freq, x, y

    def __len__(self):
        return len(self.freqs)

In [53]:
class CQTSineDataModule(L.LightningDataModule):
    def __init__(self, batch_size, num_workers=0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        spec = HYPERPARAMS
        dataset = SineData(spec)
        n_train = int(0.8*len(dataset))
        n_val = int(0.1*len(dataset))
        n_test = len(dataset) - n_train - n_val
        split = torch.utils.data.random_split(dataset, [n_train, n_val, n_test])
        self.train_dataset, self.val_dataset, self.test_dataset = split

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

In [54]:
class Student(L.LightningModule):
    def __init__(self, spec):
        super().__init__()
        self.spec = spec
        self.train_outputs = []
        self.test_outputs = []
        self.val_outputs = []
        self.loss = torch.nn.CosineSimilarity(dim=-1)

    def step(self, batch, fold):
        feat = batch[2]
        x = batch[1]
        outputs = self.forward(x)
        print(feat.shape)
        print(outputs.shape)
        loss = 0.5 * (1-self.loss(outputs[:,:,:], feat[:,:,1:]).mean())
        if fold == "train":
            self.train_outputs.append(loss)
        elif fold == "test":
            self.test_outputs.append(loss)
        elif fold == "val":
            self.val_outputs.append(loss)
        return {'loss': loss}
    
    def training_step(self, batch, batch_idx):
        return self.step(batch, "train")
    
    def validation_step(self, batch, batch_idx):
        return self.step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self.step(batch, "test")
    
    def on_train_epoch_start(self):
        self.train_outputs = []
        self.test_outputs = []
        self.val_outputs = []

    def on_train_epoch_end(self):
        avg_loss = torch.tensor(self.train_outputs).mean()
        self.log('train_loss', avg_loss, prog_bar=False)
    
    def on_test_epoch_end(self):
        avg_loss = torch.tensor(self.test_outputs).mean()
        self.log('test_loss', avg_loss, prog_bar=False)

    def on_validation_epoch_end(self):
        avg_loss = torch.tensor(self.val_outputs).mean()
        self.log('val_loss', avg_loss, prog_bar=False)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [55]:
class MuReNN(Student):
    def __init__(self, spec, Q_multiplier=16):
        super().__init__(spec)
        self.J = spec["n_octave"]
        self.Q = spec["bins_per_octave"]
        self.tfm = MuReNNDirect(
            J = spec["n_octave"],
            Q = spec["bins_per_octave"],
            T = Q_multiplier*spec["bins_per_octave"],
            in_channels = 1
        )
        self.save_hyperparameters()

    def forward(self, x):
        x = x.reshape(x.shape[0],1,-1)
        y_hat = self.tfm(x)
        y_hat = y_hat.reshape(y_hat.shape[0], self.J*self.Q, -1)
        y_hat = torch.flip(y_hat, dims=(-2,))
        return y_hat

In [56]:
import os
from datetime import timedelta
import datetime
import time

In [57]:
start_time = int(time.time())
sav_dir = "/Users/zhang/MuReNN/murenn_theory/reproduce_waspaa.ipynb/chpt"
domain = "synth"
job_id = 1
batch_size = 10
MAX_EPOCHS = 100 
model_dir = os.path.join(sav_dir, domain)
model_sav_path = os.path.join(model_dir, str(job_id))
os.makedirs(model_sav_path, exist_ok=True)

checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
    dirpath=model_sav_path,
    monitor="val_loss",
    save_last=True,
    filename="best",
    save_weights_only=False,
)

tb_logger = L.pytorch.loggers.TensorBoardLogger(
    save_dir=os.path.join(model_sav_path, "logs")
)

if torch.cuda.is_available():
    accelerator = "gpu"
    devices = -1
    torch.multiprocessing.set_start_method("spawn")
else:
    accelerator = "cpu"
    devices = "auto"
print("Current device: ", accelerator)

dataset = CQTSineDataModule(batch_size=batch_size)
print(str(datetime.datetime.now()) + " Finished initializing dataset")


model = MuReNN(HYPERPARAMS)
print(str(datetime.datetime.now()) + " Finished initializing model")

trainer = L.Trainer(
    accelerator=accelerator,
    devices=devices,
    max_epochs=MAX_EPOCHS,
    callbacks=[checkpoint_cb],
    logger=tb_logger,
    max_time=timedelta(hours=12),
)
trainer.fit(model, dataset)

test_loss = trainer.test(model, dataset, verbose=False)
print("Model saved at: {}".format(model_sav_path))
print("Average test loss: {}".format(test_loss))
print("\n")

print(str(datetime.datetime.now()) + " Success.")
elapsed_time = time.time() - int(start_time)
elapsed_hours = int(elapsed_time / (60 * 60))
elapsed_minutes = int((elapsed_time % (60 * 60)) / 60)
elapsed_seconds = elapsed_time % 60.0
elapsed_str = "{:>02}:{:>02}:{:>05.2f}".format(
    elapsed_hours, elapsed_minutes, elapsed_seconds
)
print("Total elapsed time: " + elapsed_str + ".")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name | Type             | Params | Mode 
--------------------------------------------------
0 | loss | CosineSimilarity | 0      | train
1 | tfm  | MuReNNDirect     | 8.2 K  | train
--------------------------------------------------
8.2 K     Trainable params
0         Non-trainable params
8.2 K     Total params
0.033     Total estimated model params size (MB)


Current device:  cpu
2024-06-19 19:19:01.417287 Finished initializing dataset
2024-06-19 19:19:01.428129 Finished initializing model
CQT kernels created, time used = 0.0088 seconds
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]torch.Size([10, 64, 65])
torch.Size([1, 64, 640])


RuntimeError: The size of tensor a (640) must match the size of tensor b (64) at non-singleton dimension 2